In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.8 MB/s eta 0:00:00


In [2]:
!pip install -q yt-dlp pydub pandas tqdm mistralai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.1/182.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.3/509.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 10.0 MB/s eta 0:00:00


In [3]:
import os
import json
import time
from google.colab import drive
from mistralai import Mistral
from tqdm.notebook import tqdm
from google.colab import userdata

print("Montando Google Drive per salvare i dati in modo sicuro...")
drive.mount('/content/drive')



print("✅ Setup completato!")

Montando Google Drive per salvare i dati in modo sicuro...
Mounted at /content/drive
✅ Setup completato!


In [ ]:
# MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')

# OUTPUT_DIR = "/content/drive/MyDrive/Mistral_Hackathon/vezstral_data"
# JSONL_PATH = f"{OUTPUT_DIR}/bologna_training_data.jsonl"

In [4]:
from huggingface_hub import login
from google.colab import userdata

# Assicurati di avere HF_TOKEN nei tuoi secrets di Colab (con permessi di WRITE)
login(token=userdata.get('HF_TOKEN'))
print("✅ Logged into Hugging Face!")

✅ Logged into Hugging Face!


In [5]:
import torch
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [6]:
import json

INPUT = "/content/drive/MyDrive/Mistral_Hackathon/vezstral_data_v2/bologna_training_data_v2.jsonl"
OUTPUT = "/content/drive/MyDrive/Mistral_Hackathon/vezstral_data_v2/bologna_training_data_v2_clean.jsonl"

FINAL_SP = """Sei Vezstral, un'AI bolognese verace, simpatica e un po' cinica.
Parli come un giovane bolognese doc: mix di italiano colloquiale e slang locale.

COMPORTAMENTO:
- A input brevi (ciao, come stai, tortellini?) rispondi BREVEMENTE, 1-2 frasi max
- A domande vere rispondi con sostanza ma senza fare il professore
- Non fare mai monologhi. Non spiegare mai lo slang che usi.
- Tono: ironico, caldo, diretto. Come un amico al Pratello, non un chatbot.
- Se non sai qualcosa, dillo alla bolognese: "soccia, brisa lo so"

SLANG NATURALE (usane 2-4 per risposta):
   - Saluti/Appellativi: Bella vez, regaz, cinno (bambino/ragazzino), umarell, biassanot (nottambulo), maraglio (tamarro/grezzo), cioccapiatti (bugiardo/pallonaro).
   - Esclamazioni: Soccia, sorbole, brisa (= "non"), a balùs (tantissimo), non c'è pezza, va' mo là, prendere un muro/muffo (rifiuto).
   - Verbi: fare balotta (fare festa), polleggiarsi (rilassarsi), intortare (flirtare), gubbiare (dormire), nasare (intuire).
   - Sostantivi: rusco (spazzatura), pilla (soldi), plumma (al verde), paglia (sigaretta), biga (bici), ferro (auto), branda (letto), tange (tangenziale), ciappino (lavoretto).
   - Aggettivi: invornito/imbalzato (distratto), imbarellato (stanco/sbronzo), loffio (noioso)."""

kept = 0
skipped = 0

with open(INPUT) as fin, open(OUTPUT, "w") as fout:
    for i, line in enumerate(fin):
        try:
            obj = json.loads(line)
            msgs = obj["messages"]

            # Fix capital-C Content bug
            for m in msgs:
                if "Content" in m:
                    m["content"] = m.pop("Content")

            # Normalize system prompt
            if msgs[0]["role"] == "system":
                msgs[0]["content"] = FINAL_SP

            # Validate
            roles = [m["role"] for m in msgs]
            assert "user" in roles and "assistant" in roles

            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            kept += 1
        except Exception as e:
            skipped += 1
            print(f"Skipped line {i}: {e}")

print(f"✅ Clean: {kept} kept, {skipped} skipped → {OUTPUT}")

✅ Clean: 931 kept, 0 skipped → /content/drive/MyDrive/Mistral_Hackathon/vezstral_data_v2/bologna_training_data_v2_clean.jsonl


In [9]:
MODEL_ID = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
DATA_PATH = OUTPUT
OUTPUT_DIR = "/content/drive/MyDrive/Mistral_Hackathon/Vezstral-3B-V2"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Caricamento Dataset...")
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(f"✅ Trovate {len(dataset)} conversazioni per il training.")

Caricamento Dataset...
✅ Trovate 931 conversazioni per il training.


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Mistral3ForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
import torch


# ── 1. Tokenizer ──────────────────────────────────────────────────────────────
print("1. Caricamento Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── 2. Model in 4-bit, tutto fp16 ────────────────────────────────────────────
print("2. Caricamento Modello in 4-bit fp16...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # <-- fp16, not bf16
)

model = Mistral3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,              # <-- fp16, not bf16
)
model.config.use_cache = False              # required for gradient checkpointing
model = prepare_model_for_kbit_training(model)

# ── 3. LoRA adapter ───────────────────────────────────────────────────────────
print("3. Applicazione Adapter LoRA...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ── 3.5. Purge any surviving bf16 tensors (params + buffers only) ─────────────
# DO NOT touch requires_grad params here — let fp16 trainer handle them
print("3.5. Purga bf16 residui...")
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

for name, buf in model.named_buffers():
    if buf.dtype == torch.bfloat16:
        buf.data = buf.data.to(torch.float16)

# ── 4. Dataset ────────────────────────────────────────────────────────────────
print("4. Preparazione Dataset...")
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(f"✅ {len(dataset)} esempi trovati.")

def format_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return example

formatted_dataset = dataset.map(format_chat_template)

# ── 5. Trainer ────────────────────────────────────────────────────────────────
print("5. Configurazione Trainer...")
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    optim="paged_adamw_8bit",
    fp16=False,                      # <-- fp16 AMP
    bf16=False,                     # <-- bf16 OFF
    save_strategy="epoch",
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_length=512,             # SFTConfig uses max_seq_length, not max_length
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    args=training_args,
)



1. Caricamento Tokenizer...
2. Caricamento Modello in 4-bit fp16...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

3. Applicazione Adapter LoRA...
trainable params: 12,517,376 || all params: 3,861,607,424 || trainable%: 0.3241
3.5. Purga bf16 residui...
4. Preparazione Dataset...
✅ 931 esempi trovati.


Map:   0%|          | 0/931 [00:00<?, ? examples/s]

5. Configurazione Trainer...


Tokenizing train dataset:   0%|          | 0/931 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/931 [00:00<?, ? examples/s]

In [13]:
# ── 6. Train ──────────────────────────────────────────────────────────────────
print("🚀 Partenza Addestramento!")
trainer.train()

print("✅ Salvataggio adapter LoRA...")
trainer.model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print("✅ Fatto!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 2}.


🚀 Partenza Addestramento!


Step,Training Loss
10,2.131332
20,0.427427
30,0.300446
40,0.274862
50,0.252903
60,0.235268
70,0.236916
80,0.241759
90,0.240860
100,0.222643


KeyboardInterrupt: 

In [14]:
trainer.model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print("✅ Salvato!")

✅ Salvato!


In [ ]:
# from trl import SFTConfig, SFTTrainer

# print("Preparazione del Trainer...")

# # Funzione per formattare i messaggi nello standard chat template
# def format_chat_template(example):
#     example["text"] = tokenizer.apply_chat_template(example["messages"], tokenize=False)
#     return example

# formatted_dataset = dataset.map(format_chat_template)

# training_args = SFTConfig(
#     output_dir=OUTPUT_DIR,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     logging_steps=10,
#     num_train_epochs=3,
#     optim="paged_adamw_8bit",
#     fp16=True,
#     save_strategy="epoch",
#     gradient_checkpointing=True,
#     dataset_text_field="text",
#     max_length=512,
# )

# trainer = SFTTrainer(
#     model=model,
#     train_dataset=formatted_dataset,
#     # HO RIMOSSO peft_config DA QUI!
#     processing_class=tokenizer,
#     args=training_args,
# )

# print("🚀 Partenza Addestramento! Goditi lo spettacolo.")
# trainer.train()

# print("✅ Addestramento Completato! Salvataggio dell'adapter LoRA...")
# trainer.model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
# tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")

In [2]:
# 1. Montiamo Google Drive per accedere al modello salvato
from google.colab import drive
drive.mount('/content/drive')

# 2. Facciamo il login su Hugging Face usando il tuo token segreto
from huggingface_hub import login, HfApi
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

# 3. Definiamo la cartella dove si trova l'adapter
OUTPUT_DIR = "/content/drive/MyDrive/Mistral_Hackathon/Vezstral-3B-V2"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:


# 4. Inizializziamo l'API e carichiamo!
api = HfApi()

REPO_NAME = "n1kg0r/Vezstral-3B-Bolognese-v2"

print(f"Creazione della repository {REPO_NAME} su Hugging Face...")
api.create_repo(repo_id=REPO_NAME, exist_ok=True)

print("Caricamento dell'adapter in corso...")
api.upload_folder(
    folder_path=f"{OUTPUT_DIR}/final_adapter",
    repo_id=REPO_NAME,
    repo_type="model",
)
print(f"✅ Fatto! Vezstral è live su: https://huggingface.co/{REPO_NAME}")

Creazione della repository n1kg0r/Vezstral-3B-Bolognese-v2 su Hugging Face...
Caricamento dell'adapter in corso...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...al_adapter/tokenizer.json:   1%|1         |  209kB / 17.1MB            

  ...adapter_model.safetensors:   2%|2         |  565kB / 25.1MB            

✅ Fatto! Vezstral è live su: https://huggingface.co/n1kg0r/Vezstral-3B-Bolognese-v2
